# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, a clutch term, form, and consistency. A season they missed counts as zero. Every career rate is divided by the same five-season total, 4.0951. The weights are 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or a super over. A wide or a no-ball adds one conceded run. Bye and leg-bye amounts are not in the file, and neither are extra runs beyond that one-run penalty, so a team total is still a lower bound. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied.

A legal ball is not a wide, a no-ball, or a super over. Pressure is the occasion times the competition times the ball, plus a shock from the last six balls and a break from the previous eighteen. The current ball is not included. The shock starts with dots and wickets for a batter, and fours and sixes for a bowler, then adds the wickets above xW and the runs above xR on those same six balls. Bowling bands are cut inside pace and inside spin, so the new ball does not put more pace bowlers into the high band. A recognized batter is anyone at positions 1 to 7. Zero, medium, and high are the bottom half, the next 30 percent, and the top fifth of that pressure.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal, excluding super overs", len(legal))
print("five-season weight total", round(ratings.W_SUM, 4))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))

deliveries 200999 legal, excluding super overs 192675
five-season weight total 4.0951


,balls,share
band,,
zero,94837,49.2
medium,59221,30.7
high,38617,20.0


## Elo

Team strength updates after each match. Batting strength and bowling strength update after each legal ball, on separate IPL and international ladders. A player with few balls is held near 1500, and that hold loosens as the career lengthens. Venue strength uses only balls already bowled at that ground. A ball is worth the runs and the wicket chance minus what those ratings expected.

In [2]:
mu, p0, rho = ratings.expectation_tables(legal)
legal = ratings.fit_elo(legal, mu, p0)
summary = legal.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))
print("mean bat runs per legal ball", round(rho, 3))

k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 917.5
batter_elo_max: 2566.0
batter_elo_sd: 146.8
bowler_elo_min: 1019.6
bowler_elo_max: 2125.4
bowler_elo_sd: 125.8
team_elo_min: 1361.3
team_elo_max: 1647.0
grounds: 97
step_sizes: as written
mean bat runs per legal ball 1.394


## Batter scores

Baseline is strike rate and average against the phases that batter actually plays. The three bands are the strength-adjusted ball value inside zero, medium, and high pressure. Peers compare an innings with teammates and the opposition. Clutch is that comparison in the high band. Form uses up to eight earlier innings and is pulled toward the middle when that sample is short. Consistency prefers a narrower spread of match values.

Each term is a z-score on players with at least 30 decay-weighted legal balls, and each z-score stops at 3. That group of 30 is the reference for all eight terms. The composite is 0.20 baseline, 0.10 zero, 0.15 medium, 0.20 high, 0.15 clutch, 0.10 peers, 0.05 form, and 0.05 consistency. The extra 0.05, which the eight original shares did not account for, is on the baseline.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 223


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Tim David,middle order,1.473,2.211,1.565,2.193,0.413,2.489,0.869,0.960,-0.908,2153,177.202,34.726,1002.709,314.757
1,2.0,Vaibhav Sooryavanshi,top order,1.467,2.653,1.844,2.064,0.663,1.219,2.904,-0.306,-2.960,1509,223.887,43.114,652.100,45.700
2,3.0,Maaz Sadaqat,top order,1.245,2.642,1.089,0.368,1.270,-0.466,2.732,0.865,1.034,404,177.193,80.800,213.500,37.700
3,4.0,Cameron Green,middle order,1.229,1.681,1.424,2.361,0.164,1.517,0.212,2.183,0.106,1631,152.430,38.833,903.103,182.020
4,5.0,Josh Inglis,top order,1.227,1.410,1.361,3.000,-0.109,2.742,0.644,-0.717,-1.180,1465,162.237,30.521,762.222,148.819
5,6.0,Jos Buttler,top order,1.211,2.064,0.969,0.086,0.554,2.991,1.523,-0.238,-0.223,4384,155.461,41.752,2303.778,286.609
6,7.0,Heinrich Klaasen,middle order,1.072,2.315,1.803,1.164,-1.240,3.000,1.267,-1.481,-0.016,2356,167.092,39.932,1199.960,272.150
7,8.0,Shashank Singh,middle order,1.045,1.772,1.475,0.922,-0.076,2.404,0.897,1.386,-1.991,905,161.607,37.708,480.187,160.029
8,9.0,Mahendra Singh Dhoni,lower order,1.044,0.923,1.339,1.801,0.700,2.008,1.267,0.153,-2.412,690,150.000,31.364,352.416,128.987
9,10.0,Andre Russell,lower order,1.038,1.477,1.356,1.623,0.061,2.182,0.899,0.326,-1.633,1237,170.621,28.114,553.096,196.604


roles in the batter top 10
role
middle order    4
top order       4
lower order     2


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's pace or spin and the phases they bowl. A lower rate is a higher score. The high band, clutch, and peers use the same direction.

In [4]:
bowlers = ratings.score_bowlers(legal, work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 232


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Jasprit Bumrah,Pace,1.493,1.256,1.611,2.441,1.289,0.677,2.867,1.409,-0.037,85.0,6.920,20.106,1455.083,307.682
1,2.0,Matthew Humphreys,Spin,1.340,1.189,1.618,3.000,1.135,0.367,1.433,0.608,0.687,20.0,6.866,15.250,277.028,31.330
2,3.0,Nathan Ellis,Pace,1.331,0.949,1.991,0.790,2.656,0.810,1.268,0.056,0.822,58.0,7.956,15.810,736.037,156.962
3,4.0,Noor Ahmad,Spin,1.090,0.503,2.453,2.932,-0.039,1.348,1.281,-0.116,-0.245,85.0,7.640,17.259,1281.203,283.918
4,5.0,Benjamin Sears,Pace,1.065,0.953,0.091,2.027,1.590,0.632,0.112,2.392,0.346,30.0,7.921,15.100,386.485,96.573
5,6.0,George Linde,Spin,1.020,0.324,0.232,0.341,1.217,3.000,0.915,1.292,0.625,15.0,7.611,18.867,263.080,42.410
6,7.0,Shoriful Islam,Pace,0.987,0.539,-1.009,2.054,3.000,0.919,-0.358,-1.235,0.640,41.0,7.972,19.293,650.221,87.655
7,8.0,Abrar Ahmed,Spin,0.968,1.332,0.835,0.471,1.026,1.375,1.525,-0.420,0.090,46.0,6.565,16.609,689.250,126.370
8,9.0,Eshan Malinga,Pace,0.934,0.411,0.612,0.937,1.834,0.752,0.561,2.103,0.184,55.0,8.581,16.782,874.650,213.040
9,10.0,Jacob Duffy,Pace,0.898,0.840,1.066,-0.737,3.000,-0.063,0.775,2.407,-1.090,62.0,8.353,14.145,789.222,142.324


roles in the bowler top 10
role
Pace    6
Spin    4


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 30 of those in the high band. Anyone under the line stays in the full table with no rank. The two lists are saved under `ratings/`.

In [5]:
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: season denominator", round(ratings.W_SUM, 4))
print("decision: elo step sizes", summary["step_sizes"])

wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: season denominator 4.0951
decision: elo step sizes as written
